# **Exploratory Data Analysis**

In [ ]:
# Standard library imports
import glob
import os

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from tqdm import tqdm
import pylab
import seaborn as sns

# 3. Local/module specific imports

### **1. Dataset Overview**

* **Size of the Dataset**: Determine the number of images in your dataset. A larger dataset can help in training a more robust model but may require more computational resources.

In [ ]:
df = pd.read_csv('../data/stage_2_train_labels.csv')
print(f"Dataset shape: {df.shape}")

* **Class Distribution**: Check the distribution of images across the two classes (pneumonia and normal). Imbalanced datasets can lead to biased models, so you might need techniques like data augmentation, oversampling, or class weighting during training.

In [ ]:
df_detailed = pd.read_csv('../data/stage_2_detailed_class_info.csv')

# Count unique patients per class
class_counts = df_detailed.groupby('class')['patientId'].nunique()

print("\nClass Distribution (Unique Patients):")
print("-" * 35)
for class_name, count in class_counts.items():
    print(f"{class_name}: {count} patients")
print("-" * 35)
print(f"Total unique patients: {df_detailed['patientId'].nunique()}")

* **Diversity**: Ensure that the dataset includes a diverse range of cases (e.g., different stages of pneumonia, various patient demographics) to avoid biases and improve generalization.

In [ ]:
# Let's analyze patient demographics from DICOM metadata
def extract_patient_info(patient_id):
    """Extract patient demographic information from DICOM metadata"""
    dcm_file = f'../data/stage_2_train_images/{patient_id}.dcm'
    dcm_data = pydicom.dcmread(dcm_file)
    
    # Extract age and gender if available
    age = getattr(dcm_data, 'PatientAge', 'Unknown')
    gender = getattr(dcm_data, 'PatientSex', 'Unknown')
    
    return {
        'patientId': patient_id,
        'age': age,
        'gender': gender
    }

# Process all patients instead of just a sample
all_patients = df['patientId'].unique()

# Extract demographic information
print("Extracting patient demographics from DICOM metadata...")
patient_demographics = []
for patient_id in tqdm(all_patients):
    patient_info = extract_patient_info(patient_id)
    patient_demographics.append(patient_info)

# Create a DataFrame with the demographic information
demographics_df = pd.DataFrame(patient_demographics)

# Merge with class information
merged_df = demographics_df.merge(df_detailed[['patientId', 'class']].drop_duplicates(), 
                                 on='patientId', how='left')

# Clean age data (remove 'Y' if present and convert to numeric)
merged_df['age_numeric'] = merged_df['age'].str.replace('Y', '').astype(float)

# Filter out unrealistic age values (e.g., above 100)
merged_df = merged_df[merged_df['age_numeric'] <= 100]

# Display demographic statistics by class
print("\nDemographic Statistics by Class:")
print("-" * 50)
for class_name, group in merged_df.groupby('class'):
    print(f"Class: {class_name}")
    print(f"Count: {len(group)}")
    
    # Gender distribution
    gender_counts = group['gender'].value_counts()
    print(f"Gender distribution: {gender_counts.to_dict()}")
    
    # Age statistics
    age_stats = group['age_numeric'].describe()
    print(f"Age statistics: Mean={age_stats['mean']:.1f}, Median={age_stats['50%']:.1f}, Min={age_stats['min']:.1f}, Max={age_stats['max']:.1f}")
    print("-" * 50)

# Define colorblind-friendly colors that work well on dark backgrounds
# Using a high-contrast palette with blue and orange
colors = ['#1E88E5', '#FFA726', '#4CAF50', '#E53935']  # Blue, Orange, Green, Red - added more colors to handle more classes

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot age distribution on the first subplot
for i, (class_name, group) in enumerate(merged_df.groupby('class')):
    # Make sure we don't exceed the number of colors in our list
    color_idx = i % len(colors)
    sns.histplot(group['age_numeric'].dropna(), label=class_name, 
                 color=colors[color_idx], alpha=0.7, kde=True, ax=ax1)
ax1.set_title('Age Distribution by Class', color='white')
ax1.set_xlabel('Age (years)', color='white')    # x-axis label for age plot
ax1.set_ylabel('Count', color='white')          # y-axis label for age plot
ax1.legend()
ax1.tick_params(colors='white')

# Set x-axis limits to reasonable age range
ax1.set_xlim(0, 100)

# Plot gender distribution on the second subplot
gender_class = pd.crosstab(merged_df['gender'], merged_df['class'])
gender_class.plot(kind='bar', stacked=False, ax=ax2, color=colors[:len(gender_class.columns)])
ax2.set_title('Gender Distribution by Class', color='white')
ax2.set_xlabel('Gender', color='white')         # x-axis label for gender plot
ax2.set_ylabel('Count', color='white')          # y-axis label for gender plot
ax2.tick_params(colors='white')
ax2.legend(framealpha=0.8)

# Adjust layout and display
plt.tight_layout()
plt.show()

### **2. Image Quality and Characteristics**

* **Resolution and Dimensions**: Analyze the resolution and dimensions of the X-ray images. Consistent image sizes are typically required for training convolutional neural networks (CNNs), so you might need to resize images.

In [ ]:
# Get first patient ID and load corresponding DICOM file
patientId = df['patientId'][0]
dcm_file = f'../data/stage_2_train_images/{patientId}.dcm'
dcm_data = pydicom.dcmread(dcm_file)

# Extract pixel array and display image properties
image = dcm_data.pixel_array
print(f"Data type: {image.dtype}")
print(f"Image dimensions (height x width): {image.shape}")

This was for one image. Let's make sure this is the case for every image.

In [ ]:
check = False

if check:
    # Check all images for consistent dimensions and data type
    image_shapes = set()
    image_dtypes = set()

    print("Checking all images...")
    for patientId in tqdm(df['patientId']):
        dcm_file = f'../data/stage_2_train_images/{patientId}.dcm'
        dcm_data = pydicom.dcmread(dcm_file)
        image = dcm_data.pixel_array
        
        image_shapes.add(image.shape)
        image_dtypes.add(str(image.dtype))

    print("\nResults:")
    print("-" * 50)
    print(f"Number of unique image shapes: {len(image_shapes)}")
    for shape in image_shapes:
        print(f"Shape found: {shape}")

    print(f"\nNumber of unique data types: {len(image_dtypes)}")
    for dtype in image_dtypes:
        print(f"Data type found: {dtype}")

    if len(image_shapes) == 1 and len(image_dtypes) == 1:
        print("\nAll images have consistent dimensions and data type!")
    else:
        print("\nWarning: Inconsistencies found in image dimensions or data types")
else:
    print("Checking all images...")
    print("\nResults:")
    print("-" * 50)
    print("Number of unique image shapes: 1")
    print("Shape found: (1024, 1024)")
    print("\nNumber of unique data types: 1")
    print("Data type found: uint8")
    print("\nAll images have consistent dimensions and data type!")

Expected output is directly provided to spare computational time.

* **Color Channels**: Most X-rays are grayscale. Confirm this and ensure that all images are consistently formatted.

In [ ]:
image_dir = '../data/stage_2_train_images'

if check:
    # Initialize lists to store statistics
    num_channels = []
    min_values = []
    max_values = []
    mean_values = []
    std_values = []

    # Go through all images
    for filename in tqdm(os.listdir(image_dir)):
        if filename.endswith('.dcm'):
            image_path = os.path.join(image_dir, filename)
            image = pydicom.dcmread(image_path).pixel_array
            
            # Collect statistics
            num_channels.append(len(image.shape))
            min_values.append(image.min())
            max_values.append(image.max())
            mean_values.append(image.mean())
            std_values.append(image.std())

    # Print summary statistics
    print("Summary across all images:")
    print(f"Number of channels: {set(num_channels)}")
    
    # Check if images are grayscale using the same criteria as the previous cell
    is_grayscale = []
    for i, filename in enumerate(os.listdir(image_dir)):
        if filename.endswith('.dcm'):
            image_path = os.path.join(image_dir, filename)
            image = pydicom.dcmread(image_path).pixel_array
            
            is_2d = len(image.shape) == 2
            is_3d_single_channel = len(image.shape) == 3 and image.shape[2] == 1
            is_3d_identical_channels = False
            if len(image.shape) == 3 and image.shape[2] == 3:
                is_3d_identical_channels = np.array_equal(image[:,:,0], image[:,:,1]) and np.array_equal(image[:,:,0], image[:,:,2])
            
            is_grayscale.append(is_2d or is_3d_single_channel or is_3d_identical_channels)
    
    print(f"All images grayscale?: {all(is_grayscale)}")

    print("\nPixel value statistics:")
    print(f"Min value range: {min(min_values)} - {max(min_values)}")
    print(f"Max value range: {min(max_values)} - {max(max_values)}")
    print(f"Mean value range: {min(mean_values):.2f} - {max(mean_values):.2f}")
    print(f"Std value range: {min(std_values):.2f} - {max(std_values):.2f}")
else:
    print("100%|██████████| 26684/26684 [03:12<00:00, 138.63it/s]")
    print("Summary across all images:")
    print("Number of channels: {2}")
    print("All images grayscale?: True")
    print("\nPixel value statistics:")
    print("Min value range: 0 - 95")
    print("Max value range: 146 - 255") 
    print("Mean value range: 26.42 - 229.15")
    print("Std value range: 18.84 - 99.12")

The variability among pixel values (min value range reaching 95, max value range starting at 146 and mean value range vaying from 26.42 to 229.15) imply the need for normalization to have a consistent intensity ranges, which can help in faster convergence during training. Histogram equalization might also be required, but it is more case-specific.

* **Artifacts and Noise**: Look for any artifacts, noise, or inconsistencies in the images. These could affect model performance and might need to be addressed through preprocessing.

In [ ]:
import random

# Function to analyze artifacts, noise and inconsistencies
def analyze_image_quality(image):
    """
    Analyzes image quality by looking for artifacts and noise
    Returns quality metrics
    """
    # Calculate noise metrics
    noise_ratio = np.std(image) / np.mean(image) if np.mean(image) != 0 else 0
    
    # Detect regions with outlier values (potential artifacts)
    threshold = np.mean(image) + 2 * np.std(image)
    potential_artifacts = np.sum(image > threshold) / image.size
    
    # Calculate contrast with handling for edge cases
    if np.max(image) == np.min(image):
        contrast = 0
    else:
        # Normalize image to 0-1 range first to avoid overflow
        img_norm = (image - np.min(image)) / (np.max(image) - np.min(image))
        contrast = (np.max(img_norm) - np.min(img_norm)) / (np.max(img_norm) + np.min(img_norm))
    
    return noise_ratio, potential_artifacts, contrast

# Analyze a random subset of images
sample_size = min(1000, len(os.listdir(image_dir)))  # Analyze 1000 images or less
sampled_files = random.sample([f for f in os.listdir(image_dir) if f.endswith('.dcm')], sample_size)

noise_ratios = []
artifact_ratios = []
contrast_values = []
problematic_ids = []

print(f"Analyzing {sample_size} random images...")
for filename in tqdm(sampled_files):
    image_path = os.path.join(image_dir, filename)
    image = pydicom.dcmread(image_path).pixel_array
    
    noise_ratio, artifacts, contrast = analyze_image_quality(image)
    noise_ratios.append(noise_ratio)
    artifact_ratios.append(artifacts)
    contrast_values.append(contrast)

# Calculate thresholds
threshold_noise = np.percentile(noise_ratios, 95)
threshold_artifacts = np.percentile(artifact_ratios, 95)
threshold_contrast = np.percentile(contrast_values, 5)

# Identify problematic images and store their IDs
for i, filename in enumerate(sampled_files):
    patient_id = filename.replace('.dcm', '')
    if (noise_ratios[i] > threshold_noise or 
        artifact_ratios[i] > threshold_artifacts or 
        contrast_values[i] < threshold_contrast):
        problematic_ids.append(patient_id)

print("\nImage quality statistics:")
print(f"Average noise ratio: {np.mean(noise_ratios):.3f}")
print(f"Average artifact proportion: {np.mean(artifact_ratios):.3f}")
print(f"Average contrast: {np.mean(contrast_values):.3f}")

print("\nPatient IDs with potential quality issues:")
for pid in problematic_ids:
    print(pid)
print(f"\nTotal problematic images found: {len(problematic_ids)}")


Let's go through all the images that were considered problematic by the algorithm and review what kind of artifact they present.

In [ ]:
# Configure plot style
plt.style.use('dark_background')  # Use dark background to reduce brightness

# Function to display a set of problematic images
def plot_problematic_batch(problematic_ids, start_idx=0, n_rows=3, n_cols=3):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 15), facecolor='black')
    axes = axes.ravel()
    
    # Get current batch of IDs
    batch_size = n_rows * n_cols
    current_batch = problematic_ids[start_idx:start_idx + batch_size]
    
    for i, ax in enumerate(axes):
        if i < len(current_batch):
            # Load and display image
            image_path = os.path.join(image_dir, f"{current_batch[i]}.dcm")
            image = pydicom.dcmread(image_path).pixel_array
            
            # Calculate quality metrics
            noise, artifacts, contrast = analyze_image_quality(image)
            
            ax.imshow(image, cmap='gray')
            ax.set_title(f'ID: {current_batch[i][:8]}...\nNoise: {noise:.2f}\nArtifacts: {artifacts:.2f}\nContrast: {contrast:.2f}', color='white')
            ax.axis('off')
        else:
            ax.axis('off')
            ax.set_facecolor('black')
    
    plt.tight_layout()
    return start_idx + batch_size

# Global variable to keep track of index
current_start_idx = 0


In [ ]:
# Mostrar el siguiente conjunto de imágenes problemáticas
current_start_idx = plot_problematic_batch(problematic_ids, current_start_idx)

We will also select a random pool of other images for a more consistent check.

In [ ]:
# Get list of all image files in directory
all_image_files = [f[:-4] for f in os.listdir(image_dir) if f.endswith('.dcm')]

# Randomly sample 9 image IDs
random_sample_ids = random.sample(all_image_files, 9)

# Display the random sample of images
def plot_random_batch(image_ids, n_rows=3, n_cols=3):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 15), facecolor='black')
    axes = axes.ravel()
    
    for i, (ax, img_id) in enumerate(zip(axes, image_ids)):
        # Load and display image
        image_path = os.path.join(image_dir, f"{img_id}.dcm")
        image = pydicom.dcmread(image_path).pixel_array
        
        # Calculate quality metrics
        noise, artifacts, contrast = analyze_image_quality(image)
        
        ax.imshow(image, cmap='gray')
        ax.set_title(f'ID: {img_id[:8]}...\nNoise: {noise:.2f}\nArtifacts: {artifacts:.2f}\nContrast: {contrast:.2f}', color='white')
        ax.axis('off')
    
    plt.tight_layout()

# Display random sample
plot_random_batch(random_sample_ids)


### **3. Annotation Quality**

* **Localization Labels**: Since your model needs to localize pneumonia opacities, check if your dataset includes bounding boxes or segmentation masks for these regions. The quality and accuracy of these annotations are crucial for training an effective localization model.

In [ ]:
# Load the train_labels.csv file
train_labels_path = '../data/stage_2_train_labels.csv'
try:
    train_df = pd.read_csv(train_labels_path)
except FileNotFoundError:
    print(f"File not found: {train_labels_path}")

# Calculate statistics for bounding box dimensions
print("\nBounding box statistics for pneumonia cases:")
pneumonia_boxes = train_df[train_df['Target'] == 1].copy()  # Use .copy() to avoid SettingWithCopyWarning
box_stats = pneumonia_boxes[['x', 'y', 'width', 'height']].describe()
print(box_stats)

# Calculate aspect ratios and areas for analysis
pneumonia_boxes.loc[:, 'aspect_ratio'] = pneumonia_boxes['width'] / pneumonia_boxes['height']
pneumonia_boxes.loc[:, 'area'] = pneumonia_boxes['width'] * pneumonia_boxes['height']

# Calculate distance from center
# First, calculate the center of each bounding box
pneumonia_boxes.loc[:, 'box_center_x'] = pneumonia_boxes['x'] + pneumonia_boxes['width'] / 2
pneumonia_boxes.loc[:, 'box_center_y'] = pneumonia_boxes['y'] + pneumonia_boxes['height'] / 2

# Assuming standard image dimensions (we could load each image to get exact dimensions)
# Using typical chest X-ray dimensions as an approximation
IMAGE_WIDTH, IMAGE_HEIGHT = 1024, 1024
IMAGE_CENTER_X, IMAGE_CENTER_Y = IMAGE_WIDTH / 2, IMAGE_HEIGHT / 2

# Calculate distance from image center to box center
pneumonia_boxes.loc[:, 'center_distance'] = np.sqrt(
    (pneumonia_boxes['box_center_x'] - IMAGE_CENTER_X)**2 + 
    (pneumonia_boxes['box_center_y'] - IMAGE_CENTER_Y)**2
)

# Identify potential outliers in box dimensions
area_threshold_high = pneumonia_boxes['area'].quantile(0.95)
area_threshold_low = pneumonia_boxes['area'].quantile(0.05)
aspect_threshold_high = pneumonia_boxes['aspect_ratio'].quantile(0.95)
aspect_threshold_low = pneumonia_boxes['aspect_ratio'].quantile(0.05)
distance_threshold_high = pneumonia_boxes['center_distance'].quantile(0.95)

# Find outliers
outliers = pneumonia_boxes[
    (pneumonia_boxes['area'] > area_threshold_high) | 
    (pneumonia_boxes['area'] < area_threshold_low) |
    (pneumonia_boxes['aspect_ratio'] > aspect_threshold_high) |
    (pneumonia_boxes['aspect_ratio'] < aspect_threshold_low) |
    (pneumonia_boxes['center_distance'] > distance_threshold_high)
]

print(f"\nFound {len(outliers)} potential outlier annotations")

# Define colorblind-friendly colors for dark background
# Using a blue and orange palette that works well for most types of color blindness
blue_color = '#1e88e5'    # Bright blue
orange_color = '#ff9800'  # Bright orange
red_color = '#f44336'     # Bright red for emphasis
green_color = '#4caf50'   # Bright green

# Plot distribution of bounding box dimensions
plt.figure(figsize=(12, 4), facecolor='black')

plt.subplot(1, 2, 1)
plt.hist(pneumonia_boxes['area'], bins=30, color=blue_color, edgecolor='white', alpha=0.8)
plt.title('Distribution of Bounding Box Areas', color='white')
plt.xlabel('Area (pixels²)', color='white')
plt.ylabel('Frequency', color='white')
plt.axvline(area_threshold_high, color=red_color, linestyle='--', linewidth=2, label='95th percentile')
plt.axvline(area_threshold_low, color=green_color, linestyle='--', linewidth=2, label='5th percentile')
plt.legend(facecolor='black', edgecolor='white', labelcolor='white')
plt.tick_params(colors='white')
plt.grid(True, color='gray', alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(pneumonia_boxes['aspect_ratio'], bins=30, color=orange_color, edgecolor='white', alpha=0.8)
plt.title('Distribution of Aspect Ratios', color='white')
plt.xlabel('Width/Height Ratio', color='white')
plt.ylabel('Frequency', color='white')
plt.axvline(aspect_threshold_high, color=red_color, linestyle='--', linewidth=2, label='95th percentile')
plt.axvline(aspect_threshold_low, color=green_color, linestyle='--', linewidth=2, label='5th percentile')
plt.legend(facecolor='black', edgecolor='white', labelcolor='white')
plt.tick_params(colors='white')
plt.grid(True, color='gray', alpha=0.3)

plt.tight_layout()

# Plot distribution of distance from center
plt.figure(figsize=(10, 5), facecolor='black')
plt.hist(pneumonia_boxes['center_distance'], bins=30, color=blue_color, edgecolor='white', alpha=0.8)
plt.title('Distribution of Bounding Box Distance from Image Center', color='white')
plt.xlabel('Distance (pixels)', color='white')
plt.ylabel('Frequency', color='white')
plt.axvline(distance_threshold_high, color=red_color, linestyle='--', linewidth=2, label='95th percentile')
plt.legend(facecolor='black', edgecolor='white', labelcolor='white')
plt.tick_params(colors='white')
plt.grid(True, color='gray', alpha=0.3)
plt.tight_layout()

# Display a few outlier examples
if len(outliers) > 0:
    # Select up to 6 outliers to display
    display_outliers = outliers.sample(min(6, len(outliers)))
    
    # Parse the data for visualization
    parsed_data = parse_data(train_df)


* **Consistency**: Ensure that the annotations are consistent across the dataset. Inconsistent labeling can confuse the model during training.

We will visually check all the potential outliers resulting from the previous analysis to make sure they are correct. 

In [ ]:
# Display the outliers
def display_outlier_examples(outliers, parsed_data, max_examples=6):
    plt.figure(figsize=(15, 10))
    display_outliers = outliers.sample(min(max_examples, len(outliers)))
    
    for i, (_, row) in enumerate(display_outliers.iterrows()):
        if i >= max_examples:  # Limit to max_examples
            break
            
        patient_id = row['patientId']
        if patient_id in parsed_data:
            plt.subplot(2, 3, i+1)
            patient_data = parsed_data[patient_id]
            draw(patient_data)
            plt.title(f"ID: {patient_id[:8]}...\nArea: {row['area']:.0f}\nRatio: {row['aspect_ratio']:.2f}")
            
    plt.tight_layout()
    plt.suptitle("Examples of Outlier Bounding Box Annotations", fontsize=16)
    plt.subplots_adjust(top=0.9)
    plt.show()

# Call the function to display current outliers
if len(outliers) > 0:
    display_outlier_examples(outliers, parsed_data)

In [ ]:
# Check if any bounding boxes exceed image dimensions
def check_bbox_dimensions(df, img_size=(1024, 1024)):
    """
    Check if any bounding boxes exceed the image dimensions.
    
    Args:
        df: DataFrame with bounding box annotations
        img_size: Tuple of (width, height) for the images
    
    Returns:
        DataFrame with problematic bounding boxes
    """
    img_width, img_height = img_size
    
    # Create a copy to avoid modifying the original dataframe
    df_check = df.copy()
    
    # Calculate the right and bottom coordinates of each box
    df_check['x_right'] = df_check['x'] + df_check['width']
    df_check['y_bottom'] = df_check['y'] + df_check['height']
    
    # Find boxes that exceed image dimensions
    problematic_boxes = df_check[
        (df_check['x'] < 0) | 
        (df_check['y'] < 0) | 
        (df_check['x_right'] > img_width) | 
        (df_check['y_bottom'] > img_height)
    ]
    
    return problematic_boxes

# Check for problematic bounding boxes
problematic_boxes = check_bbox_dimensions(train_df)

# Display summary
print(f"Found {len(problematic_boxes)} bounding boxes that exceed image dimensions")
print("\nSummary of problematic boxes:")
if len(problematic_boxes) > 0:
    print(problematic_boxes[['patientId', 'x', 'y', 'width', 'height', 'x_right', 'y_bottom']].head(10))
    
    # Visualize some problematic examples
    if 'parsed_data' in locals():
        print("\nVisualizing examples of problematic bounding boxes:")
        display_outlier_examples(problematic_boxes, parsed_data, max_examples=6)
else:
    print("No problematic bounding boxes found. All annotations are within image dimensions.")


This is the end of the exploratory data analysis. In the next script we will proceed with the preprocessing and model training.